# Exploratory data analysis

The goal of this notebook is to profile the data and see if any data cleaning or pre processing steps are required.

## Bronze to Silver - Data cleaning and basic pre-processing

Core idea of this layer: adress all the potential issues flagged by the data profiling library.

In [1]:
import os
import sys
import time
sys.path.append("..")

import pandas as pd
import polars as pl
from data_profiling import ProfileReport
from ingest import get_engine

/Users/aleksandrmedvedev/Desktop/Repositories/olist_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
engine = get_engine()

print("Waking up Azure SQL...")
for attempt in range(1, 11):
    try:
        with engine.connect() as conn:
            conn.exec_driver_sql("SELECT 1")
        print("Database is ready.")
        break
    except Exception:
        print(f"Attempt {attempt}/10 — retrying in 10s...")
        time.sleep(10)
else:
    raise RuntimeError("Database did not respond after 10 attempts.")

Waking up Azure SQL...
Database is ready.


In [3]:
customers            = pl.read_database("SELECT * FROM olist_customers_dataset", engine)
geolocation          = pl.read_database("SELECT * FROM olist_geolocation_dataset", engine)
order_items          = pl.read_database("SELECT * FROM olist_order_items_dataset", engine)
order_payments       = pl.read_database("SELECT * FROM olist_order_payments_dataset", engine)
order_reviews        = pl.read_database("SELECT * FROM olist_order_reviews_dataset", engine)
orders               = pl.read_database("SELECT * FROM olist_orders_dataset", engine)
products             = pl.read_database("SELECT * FROM olist_products_dataset", engine)
sellers              = pl.read_database("SELECT * FROM olist_sellers_dataset", engine)
category_translation = pl.read_database("SELECT * FROM product_category_name_translation", engine)

In [4]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation,
}

In [5]:
dataset_reports = {}

for name, df in datasets.items():
    report = ProfileReport(df.to_pandas(), title=f"{name} profiling report", minimal=True)
    report.to_file(f"html_reports/{name}.html")
    dataset_reports[name] = report
    print(f"Generated: html_reports/{name}.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 473.72it/s]


Generated: html_reports/customers.html


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 549.50it/s]


Generated: html_reports/geolocation.html


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 419.47it/s]


...

Generated: html_reports/products.html


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 544.64it/s]


Generated: html_reports/sellers.html


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 489.47it/s]

Generated: html_reports/category_translation.html


In [6]:
# TODO: implement transformations based on profiling report alerts
customers_cleaned            = customers
geolocation_cleaned          = geolocation
order_items_cleaned          = order_items
order_payments_cleaned       = order_payments
order_reviews_cleaned        = order_reviews
orders_cleaned               = orders
products_cleaned             = products
sellers_cleaned              = sellers
category_translation_cleaned = category_translation

In [7]:
cleaned_datasets = {
    "customers": customers_cleaned,
    "geolocation": geolocation_cleaned,
    "order_items": order_items_cleaned,
    "order_payments": order_payments_cleaned,
    "order_reviews": order_reviews_cleaned,
    "orders": orders_cleaned,
    "products": products_cleaned,
    "sellers": sellers_cleaned,
    "category_translation": category_translation_cleaned,
}

for name, df in cleaned_datasets.items():
    before = dataset_reports[name]
    after  = ProfileReport(df.to_pandas(), title=f"{name} - after", minimal=True)
    comparison = before.compare(after)
    comparison.to_file(f"html_reports/comparison_reports/{name}_cleaned_comparison.html")
    print(f"Generated: html_reports/comparison_reports/{name}_cleaned_comparison.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 815.06it/s]


Generated: html_reports/comparison_reports/customers_cleaned_comparison.html


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 607.08it/s]


Generated: html_reports/comparison_reports/geolocation_cleaned_comparison.html


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 502.73it/s]


...

Generated: html_reports/comparison_reports/products_cleaned_comparison.html


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 609.02it/s]


Generated: html_reports/comparison_reports/sellers_cleaned_comparison.html


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 630.91it/s]

Generated: html_reports/comparison_reports/category_translation_cleaned_comparison.html


## Silver to Gold - Reshaping the data for business analysis (Power BI)

Core idea of this layer: remove data redundancy and reshape the data into a star schema to support the business analysis.